In [1]:
import gcsfs
import geopandas as gpd
import google.auth
import outlier_detection
import pandas as pd
import report_utils
from gtfs_curator_utils import utils
from shared_vars import PREDICTIONS_GCS, RT_MSA_DICT

credentials, project = google.auth.default()

/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.11.10) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.11.10) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)
/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-pack

Can `outlier_detection.drop_outliers` be used, similar to route cleaning, instead of `outlier_detection.remove_outliers`?

Do a sanity check to see if the prediction counts are within reasonable bounds.

Yes, these counts are ok.

In [2]:
stop_gdf = gpd.read_parquet(
    f"{PREDICTIONS_GCS}monthly_tu_stops.parquet",
    storage_options={"token": credentials.token},
)

In [3]:
stop_gdf.dtypes

month_first_day                              datetime64[ns]
month                                                 Int64
year                                                  Int64
day_type                                             object
schedule_name                                        object
schedule_base64_url                                  object
tu_base64_url                                        object
tu_name                                              object
stop_id                                              object
stop_key                                             object
avg_prediction_error_sec                            float64
n_tu_accurate_minutes                                 Int64
n_tu_complete_minutes                                 Int64
n_tu_minutes_available                                Int64
avg_prediction_spread_minutes                       float64
n_predictions                                         Int64
n_predictions_early                     

In [4]:
stop_gdf2 = gpd.read_parquet(
    f"{PREDICTIONS_GCS}monthly_tu_stops_cleaned.parquet",
    storage_options={"token": credentials.token},
)

In [8]:
stop_gdf2 = stop_gdf2.assign(
    pct_early = stop_gdf2.n_predictions_early.divide(stop_gdf.n_predictions),
    pct_ontime = stop_gdf2.n_predictions_ontime.divide(stop_gdf.n_predictions),
    pct_late = stop_gdf2.n_predictions_late.divide(stop_gdf.n_predictions)
)
stop_gdf2["pct"] = stop_gdf2[["pct_early", "pct_ontime", "pct_late"]].sum(axis=1)

In [9]:
stop_gdf2.pct.value_counts()

pct
1.0    1478334
1.0       2425
1.0         45
Name: count, dtype: Int64